# 5-Amaliy mashg‘ulot
## Mavzu: Ma’lumotlarni normallashtirish va standartlashtirish usullarini qo‘llash
**Bajaruvchi:** Talaba
**Fani:** Data Mining

---
### Ishning maqsadi:
Belgilarni bir xil masshtabga keltirishning asosiy usullarini qo‘lda va Python’da qo‘llash, ularning model natijalariga (k-NN va Qarorlar daraxti) ta’sirini tushunish.

### 1-topshiriq. Min-Max normallashtirish (qo‘lda va pandas’da)
Besh nafar xodimning yoshi berilgan: 20, 35, 50, 65, 80 yil.

In [1]:
import numpy as np
import pandas as pd

yoshlar = [20, 35, 50, 65, 80]
x_min, x_max = min(yoshlar), max(yoshlar)
farq = x_max - x_min

df1 = pd.DataFrame({'Yosh, x': yoshlar})
df1['x - x_min'] = df1['Yosh, x'] - x_min
df1["x' in [0, 1]"] = df1['x - x_min'] / farq
df1["x' in [-1, 1]"] = 2 * df1["x' in [0, 1]"] - 1

print("1-topshiriq natijaviy jadvali:")
display(df1)

# Savollarga hisob-kitoblar
yosh_42 = (42 - x_min) / farq
yosh_90 = (90 - x_min) / farq
print(f"42 yoshli xodim uchun [0, 1] dagi qiymat: {yosh_42:.3f}")
print(f"90 yoshli xodim uchun qiymat: {yosh_90:.3f} (chegaradan chiqib ketadi)")

# O'nlik masshtablash: [125, 980, -450]
vals = [125, 980, -450]
j = 3
print("O'nlik masshtablash:", {v: v / (10**j) for v in vals})

### 2-topshiriq. Z-score standartlashtirish (qo‘lda va formulalar yordamida)
Besh nafar talabaning test ballari berilgan: 60, 70, 80, 90, 100.

In [2]:
ballar = [60, 70, 80, 90, 100]
n = len(ballar)
mu = sum(ballar) / n
diffs = [x - mu for x in ballar]
diffs_sq = [(x - mu) ** 2 for x in ballar]
sigma = (sum(diffs_sq) / n) ** 0.5
z_scores = [d / sigma for d in diffs]

df2 = pd.DataFrame({
    'Ball, x': ballar,
    'x - mu': diffs,
    '(x - mu)^2': diffs_sq,
    'z': [round(z, 3) for z in z_scores]
})
print(f"mu = {mu}, sigma = {sigma:.3f}")
display(df2)

z_95 = (95 - mu) / sigma
print(f"95 ball olgan talabaning z qiymati: {z_95:.3f} (o'rtachadan {z_95:.3f}*sigma yuqori)")

### 3-topshiriq. Python’da MinMaxScaler va StandardScaler
6 nafar mijoz ma'lumotlari bo'yicha masshtablash va Evklid masofasi tahlili.

In [3]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df_bank = pd.DataFrame({
    "Yosh":    [25, 32, 47, 51, 62, 38],
    "Daromad": [4.5, 7.0, 12.0, 9.5, 6.0, 15.0]
})
display(df_bank.describe())

mm = MinMaxScaler()
df_mm = pd.DataFrame(mm.fit_transform(df_bank), columns=df_bank.columns)

st = StandardScaler()
df_st = pd.DataFrame(st.fit_transform(df_bank), columns=df_bank.columns)

print("Min-Max natijasi:")
display(df_mm.round(3))

print("Z-score natijasi:")
display(df_st.round(3))

# Evklid masofasi tahlili (1 va 5 mijoz)
p1 = df_bank.iloc[0].values
p5 = df_bank.iloc[4].values
print(f"Dastlabki masofa: {np.linalg.norm(p1 - p5):.3f}")
print(f"Min-Max dan keyin: {np.linalg.norm(df_mm.iloc[0].values - df_mm.iloc[4].values):.3f}")
print(f"Z-score dan keyin: {np.linalg.norm(df_st.iloc[0].values - df_st.iloc[4].values):.3f}")

### 4-topshiriq. Masshtablashning model natijasiga ta'siri (Wine dataset)
k-NN va Decision Tree modellarini masshtablanmagan va masshtablangan holatda solishtirish.

In [4]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline

X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

modellar = {
    "k-NN (masshtablashsiz)": KNeighborsClassifier(n_neighbors=5),
    "k-NN + MinMax": make_pipeline(MinMaxScaler(), KNeighborsClassifier(5)),
    "k-NN + Z-score": make_pipeline(StandardScaler(), KNeighborsClassifier(5)),
    "Qarorlar daraxti": DecisionTreeClassifier(random_state=42),
    "Qarorlar daraxti + Z-score": make_pipeline(StandardScaler(), DecisionTreeClassifier(random_state=42)),
}

for nom, model in modellar.items():
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    print(f"{nom:28s} : {acc:.3f} ({acc*100:.1f}%)")